# 1.二分图

## 1.1染色法判断二分图

In [ ]:
N = int(1e5 + 10)
n, m = map(int, input().split())
color = [0] * N
g = [[] for _ in range(N)]


def dfs(u, col):
    color[u] = col  # 染色当前点
    for v in g[u]:  # 遍历相邻点
        if not color[v]: # 如果没染色
            if not dfs(v, 3 - col): # 递归失败
                return False
        elif color[v] == col:   # 相邻点的颜色和当前点相同
            return False
    return True


for _ in range(m):
    a, b = map(int, input().split())
    g[a].append(b)
    g[b].append(a)
flag = True
for i in range(1, n + 1):
    if not color[i]:  # 未染色
        if not dfs(i, 1):  # 染色失败，直接退出
            flag = False
            break
if flag:
    print("Yes")
else:
    print("No")
k = 2

# 2.负权处理

## 2.1bellman

In [ ]:
N = 100010
inf = float("inf")
Edge = []  # 存放边
dist = [inf] * N

# 有边数限制的最短路
def Ballman_Ford():
    dist[1] = 0
    for _ in range(k):
        # 循环k次
        back_up = dist.copy()  # 需要进行备份， 防止更新过的点立刻影响到本次更新
        for j in range(m):      # m条边
            a, b, w = Edge[j]
            dist[b] = min(dist[b], back_up[a] + w)

    if dist[n] == inf:
        return "impossible"
    else:
        return dist[n]


n,m,k = map(int,input().split())
for _ in range(m):
    a,b,c = map(int,input().split())
    Edge.append([a,b,c])

print(Ballman_Ford())


## 2.2spfa

In [ ]:

def spfa():
    dis = [inf] * (n+ 1)
    dis[1] = 0
    s = set()
    s.add(1)      # 记录哪些点被更新过， 只有前驱结点变小了，后续的结点才能变小
    q = deque([1])
    while q:
        node = q.popleft()
        s.remove(node)     # 可以重复更新
        for y, w in g[node]:
            if dis[y] > dis[node] + w:
                dis[y] = dis[node] + w
                if y not in s:
                    q.append(y)
                    s.add(y)
    return dis[n] if dis[n] != inf else "impossible"

for _ in range(m):
    a, b, w = MII()
    g[a].append((b, w))



## 2.3 spfa判断负环

In [ ]:

n, m = MII()
g = [[] for _ in range(n + 1)]
cnt = [0] * (n + 1)
def spfa():
    dis = [0] * (n + 1)   # 初始化为0
    s = set()
    q = deque([i for i in range(1, n + 1)]) # 环的起点不确定，所以全部入队
    for i in range(1, n + 1):
        s.add(i)
    while q:
        node = q.popleft()
        s.remove(node)     # 可以重复更新
        for y, w in g[node]:
            if dis[y] > dis[node] + w:
                dis[y] = dis[node] + w
                cnt[y] = cnt[node] + 1
                if cnt[y] >= n:   # 有n条边, 说明有n+1个点, 重复了说明有环
                    return True
                if y not in s:
                    q.append(y)
                    s.add(y)
    return False

for _ in range(m):
    a, b, w = MII()
    g[a].append((b, w))
print("Yes" if spfa() else "No")



# 3.基环树

## 3.1内基环树求最大环

In [ ]:
def longestCycle(edges):
    n = len(edges)
    time = [0] * n
    clock = 1
    res = -1
    cnt = 0
    for i in range(n):
        if time[i]:continue
        start_time = clock
        cnt += 1
        x = i
        while x != -1:
            if time[x]:
                if time[x] >= start_time:
                    res = max(res, clock - time[x])
                break
            time[x] = clock
            clock += 1
            x = edges[x]
    return res


# 4.强连通分量scc

## 4.1Tarjan

In [ ]:

inf = float('inf')
n, m = MII()
N, M = int(1e4 + 10), int(5e4 + 10)
g = [[] for _ in range(N + 1)]

dfn = [0] * N  # 被访问到的实际时间点
low = [0] * N  # 能回到的最高点
id = [0] * N
timestamp = 0
stk = []
in_stk = [False] * N
scc_cnt = top = 0
Size = [0] * N
doubt = [0] * N

def tarjan(u:int):
    global timestamp, scc_cnt
    dfn[u] = low[u] = timestamp = timestamp + 1   # 时间戳，默认相等
    stk.append(u)   # 入栈
    in_stk[u] = True
    for x in g[u]:
        if not dfn[x]:  # 未访问过该结点
            tarjan(x)
            low[u] = min(low[u], low[x])   # 有可能通过该点回到过去
        elif in_stk[x]:
            low[u] = min(low[u], dfn[x])   # 在栈中我们认为栈下面的点时间戳一定是小于当前点的
    if dfn[u] == low[u]:  # 回到了自己
        scc_cnt += 1
        while True:
            y = stk.pop()  # 节点全部出栈
            in_stk[y] = False
            id[y] = scc_cnt  # 结点属于哪个强连通分量
            Size[scc_cnt] += 1  # 连通分量的大小+1
            if y == u:  # 出栈结束
                break
                
for _ in range(m):
    x, y = MII()
    g[x].append(y)

for i in range(1, n + 1):
    if not dfn[i]:
        tarjan(i)

for i in range(1, n + 1):
    for j in g[i]:
        a, b = id[i], id[j]  # scc内部互相可达
        if a != b:
            doubt[a] += 1
            
zeros = sum = 0
for i in range(1, scc_cnt + 1):
    if not doubt[i]:
        zeros += 1
        sum += Size[i]
        if zeros > 1:
            sum = 0
            break
print(sum)




# 5.拓扑

## 5.1拓扑排序

In [ ]:
# ACWING 模板
from typing import List

N = 100010
n, m = map(int, input().split())
h, e, ne = [-1] * N, [0] * N, [0] * N
idx = 0
q, d = [0] * N, [0] * N


def add(a, b):
    global idx
    e[idx] = b
    ne[idx] = h[a]
    h[a] = idx
    idx += 1


def topsort():
    hh, tt = 0, -1
    for i in range(1, n + 1):
        if d[i] == 0:
            tt += 1
            q[tt] = i
    while hh <= tt:
        t = q[hh]
        hh += 1
        i = h[t]
        while i != -1:
            j = e[i]
            d[j] -= 1
            if d[j] == 0:
                tt += 1
                q[tt] = j
            i = ne[i]
    return tt == n - 1


for _ in range(m):
    a, b = map(int, input().split())
    add(a, b)
    d[b] += 1
print(" ".join(map(str, q[:n])) if topsort() else -1)

# LC模板
from collections import deque
# n是有几个点
def topsort(edges):
    n = k = len(edges)
    g = [[] for _ in range(k)]
    left = [0] * k
    for x, y in edges:
        x -= 1
        y -= 1
        g[x].append(y)
        left[y] += 1
    order = []
    q = deque(i for i, v in enumerate(left) if v == 0)
    while q:
        x = q.popleft()
        order.append(x)
        for y in g[x]:
            left[y] -= 1
            if left[y] == 0:
                q.append(y)
    return order if len(order) == k else None


## 附上一个调用库函数的
from graphlib import TopologicalSorter


def buildMatrix(self, k: int, rowConditions: List[List[int]], colConditions: List[List[int]]) -> List[List[int]]:
    def get_pos(cons: List[List[int]]) -> List[int]:
        ts = TopologicalSorter()
        for x in range(1, k + 1):
            ts.add(x)
        for x, y in cons:
            ts.add(y, x)
        pos = [0] * (k + 1)
        for i, x in enumerate(ts.static_order()):
            pos[x] = i
        return pos

    try:
        book = {(i, j): x for x, (i, j) in enumerate(zip(get_pos(rowConditions), get_pos(colConditions)))}
        return [[book.get((i, j), 0) for j in range(k)] for i in range(k)]
    except:
        return []


# 6.最短路

## 6.1堆优化dijkstra

In [ ]:

# mlogn m:边数， n:点数
def dijkstra(start, end):
    s = set()
    dis = [inf] * (n + 1)
    dis[start] = 0
    heap = []
    heappush(heap, (0, start))  # 起始点
    while heap:
        d, node = heappop(heap)   #当前距离， 点
        if node in s:
            continue
        s.add(node)
        for y, w in g[node]:  # 更新所有点
            if dis[y] > d + w:
                dis[y] = d + w
                heappush(heap, (dis[y], y))
    return -1 if dis[end] == inf else dis[end]   # 终点


res = dijkstra()
print(res)


## 6.2Floyd

In [ ]:
'''
# O(n^3)
N = 210
d = [[float("inf")] * N for _ in range(N)]

for i in range(1, n + 1):
    dis[i][i] = 0

def floyd(): 
    for k in range(1, n + 1):      # 经过结点编号在1-k-1的最短路（插点）
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                d[i][j] = min(d[i][j], d[i][k] + d[k][j])
                
for _ in range(m):
    x, y, z = map(int, input().split())
    d[x][y] = min(d[x][y], z)  # 防止有重边
floyd()
'''

# 7.最近公共祖先

In [ ]:
from typing import *
from collections import deque



# def bit_length(x):
#     return len(bin(x)) - 2
N = int(1e4 + 10) 
g = [[] for _ in range(N)]
class TreeAncestor:
    def __init__(self):
        m = 16
        depth = [0] * N
        fa = [[-1] * m for _ in range(N)]
        def dfs(x: int, father: int) -> None:
            fa[x][0] = father
            for y in g[x]:
                if y != father:
                    depth[y] = depth[x] + 1
                    dfs(y, x)
        # dfs(0, -1)
        def bfs(root):
            q = deque([(root, -1, 0)])
            while q:
                x, father, cur_depth = q.popleft()
                depth[x] = cur_depth
                fa[x][0] = father
                for y in g[x]:
                    if y != father:
                        q.append((y, x, cur_depth + 1))


        for i in range(m - 1):
            for x in range(n):   # 下标从1开始记得改这里
                if (p := fa[x][i]) != -1:
                    fa[x][i + 1] = fa[p][i]
        self.depth = depth
        self.fa = fa

    def get_kth_ancestor(self, node: int, k: int) -> int:
        for i in range(k.bit_length()):
            if (k >> i) & 1:  # k 二进制从低到高第 i 位是 1
                node = self.fa[node][i]
        return node

    # 返回 x 和 y 的最近公共祖先（节点编号从 0 开始）
    def get_lca(self, x: int, y: int) -> int:
        if self.depth[x] > self.depth[y]:
            x, y = y, x
        # 使 y 和 x 在同一深度
        y = self.get_kth_ancestor(y, self.depth[y] - self.depth[x])
        if y == x:
            return x
        for i in range(len(self.fa[x]) - 1, -1, -1):  # 从大到小枚举，能跳就跳
            px, py = self.fa[x][i], self.fa[y][i]
            if px != py:
                x, y = px, py  # 同时上跳 2**i 步
        return self.fa[x][0]



# 8.最小生成树

## 8.1kruskal

In [ ]:

inf = float('inf')
n, m = MII()
g = []
p = [i for i in range(n + 1)]

def find(x):
    if x != p[x]:
        p[x] = find(p[x])
    return p[x]

for _ in range(m):
    a, b, w = MII()
    g.append((a, b, w))
g.sort(key=lambda x:x[2])
res = cnt = 0
for i in range(m):
    a, b, w = g[i]
    a, b = find(a), find(b)
    if a != b:
        p[a] = b
        res += w
        cnt += 1
print(res if cnt == n - 1 else "impossible")


## 8.2prim

In [ ]:

inf = float('inf')

n, m = MII()
g = [[inf] * (n + 1) for _ in range(n + 1)]  # 注意初始化为inf
dis = [inf] * (n + 1)   # 各个结点到生成树的距离

# 思路是每次找已知点的邻边最小加入即可， 然后更新

def prim():
    res = 0
    dis[1] = 0
    s = set()
    for i in range(n):
        t = -1
        for j in range(1, n + 1):       # 如果没有在树中，且到树的距离最短，则选择该点
            if j not in s and (t == -1 or dis[t] > dis[j]):
                t = j
        if i and dis[t] == inf:
            return inf
        if i:
            res += dis[t]
        s.add(t)
        for j in range(1, n + 1):  # 更新生成树外的点到生成树的距离
            dis[j] = min(dis[j], g[t][j])
    return res


for _ in range(m):
    a, b, c = MII()
    g[a][b] = g[b][a] = min(g[a][b], c)

res = prim()
print("impossible" if res == inf else res)



# 9.找环

In [ ]:
def bfs():
    s = set()
    q = deque([b])
    p = [0] * (n + 1)
    p[b] = b
    while q:
        node = q.pop()
        if node in s:
            return node
        s.add(node)
        for x in g[node]:
            if x != p[node]:
                p[x] = node
                q.append(x)
t = II()
for _ in range(t):
    n, a, b = MII()
    g = [[] for _ in range(n + 1)]
    for _ in range(n):
        u, v = MII()
        g[u].append(v)
        g[v].append(u)
    d = [-1] * (n + 1)

    root = bfs()    
    d[root] = 0
    q = deque([root])
    while q:
        node = q.popleft()
        for x in g[node]:
            if d[x] == -1:
                d[x] = d[node] + 1
                q.append(x)
    print("YES" if d[a] > d[b] else "NO")